In [18]:
# Cell 1: Setup project root (fix path)
from pathlib import Path
import os, sys

def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / "requirements.txt").exists() and (p / "api").exists():
            return p
    return cur.resolve()

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("CWD =", os.getcwd())

CWD = D:\logistics_AI


In [19]:
# Cell 2: Config
PROCESS_CODE = "TRUCKING_DELIVERY_FLOW"

DATA_DIR = "data/synth_optimal_3process_v1"
REGISTRY_DIR = "registry/synth_optimal_v1"
MODEL_DIR = "model/process_models"

# Train params
N_ESTIMATORS = 300        # nâng nhẹ từ 200 -> 300
CONTAMINATION = 0.06
RANDOM_STATE = 42

INCLUDE_CONTEXT_NUMERIC = False

In [20]:
# Cell 3: Sanity check input data
import pandas as pd
from pathlib import Path

events_path = Path(DATA_DIR) / "events.csv"
print("events_path:", events_path, "| exists:", events_path.exists())

df = pd.read_csv(events_path)
print("rows:", len(df))
print("process_code counts (top):")
print(df["process_code"].value_counts().head(10))

assert PROCESS_CODE in set(df["process_code"].unique()), f"Missing process_code in events.csv: {PROCESS_CODE}"

events_path: data\synth_optimal_3process_v1\events.csv | exists: True
rows: 692576
process_code counts (top):
process_code
TRUCKING_DELIVERY_FLOW      348000
WAREHOUSE_FULFILLMENT       276000
IMPORT_CUSTOMS_CLEARANCE     68576
Name: count, dtype: int64


In [21]:
# Cell 4: Train process model (TRUCKING)
from api.process_ai.train import train_process_model

summary = train_process_model(
    data_dir=DATA_DIR,
    registry_dir=REGISTRY_DIR,
    model_root_dir=MODEL_DIR,
    process_code=PROCESS_CODE,
    include_context_numeric=INCLUDE_CONTEXT_NUMERIC,
    n_estimators=N_ESTIMATORS,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
)

summary

{'process_code': 'TRUCKING_DELIVERY_FLOW',
 'events_rows_used': 348000,
 'cases_used': 12000,
 'include_context_numeric': False,
 'n_estimators': 300,
 'contamination': 0.06,
 'random_state': 42,
 'artifacts_dir': 'model\\process_models\\TRUCKING_DELIVERY_FLOW',
 'validation_warnings': [],
 'feature_report': {'cases': 12000,
  'dropped_cases': 0,
  'repeated_case_count': 0,
  'missing_step_cases': 0}}

In [22]:
# Cell 5: Verify artifacts saved
from pathlib import Path

art_dir = Path(MODEL_DIR) / PROCESS_CODE
print("artifacts_dir:", art_dir.resolve())
print("exists:", art_dir.exists())

sorted([p.name for p in art_dir.glob("*")])

artifacts_dir: D:\logistics_AI\model\process_models\TRUCKING_DELIVERY_FLOW
exists: True


['baselines.json',
 'feature_schema.json',
 'model.pkl',
 'scaler.pkl',
 'score_quantiles.json',
 'train_summary.json']

In [23]:
# Cell 6: Quick inference smoke test (1 case)
import pandas as pd
from pathlib import Path
from api.process_ai.inference import load_process_artifacts, analyze_case

art = load_process_artifacts(
    process_code=PROCESS_CODE,
    model_root_dir=MODEL_DIR,
    registry_dir=REGISTRY_DIR,
)

df = pd.read_csv(Path(DATA_DIR) / "events.csv")
cid = df.loc[df["process_code"] == PROCESS_CODE, "case_id"].iloc[0]
one = df[(df["process_code"] == PROCESS_CODE) & (df["case_id"] == cid)].copy()

result = analyze_case(one, art, allow_unknown_steps=False)
result

{'ok': True,
 'process_code': 'TRUCKING_DELIVERY_FLOW',
 'case_id': 'ORD_5260181590',
 'risk_score': 43.0,
 'raw_anomaly': 0.400812,
 'overall_severity': 'CRITICAL',
 'top_steps': [{'step_code': 'STEP_026_POD_UPLOADED',
   'duration_min': 91.667,
   'p95': 49.337,
   'deviation_factor': 1.858,
   'z_score': 4.912,
   'severity': 'CRITICAL'},
  {'step_code': 'STEP_008_GATE_IN_PICKUP',
   'duration_min': 19.65,
   'p95': 24.101,
   'deviation_factor': 0.815,
   'z_score': 1.184,
   'severity': 'NORMAL'},
  {'step_code': 'STEP_018_LINEHAUL_START',
   'duration_min': 274.883,
   'p95': 350.752,
   'deviation_factor': 0.784,
   'z_score': 1.138,
   'severity': 'NORMAL'},
  {'step_code': 'STEP_010_LOAD_START',
   'duration_min': 36.017,
   'p95': 49.884,
   'deviation_factor': 0.722,
   'z_score': 0.876,
   'severity': 'NORMAL'},
  {'step_code': 'STEP_020_ARRIVE_LASTMILE_DEPOT',
   'duration_min': 15.783,
   'p95': 24.233,
   'deviation_factor': 0.651,
   'z_score': 0.621,
   'severity': 'NO